In [37]:
import csv
import itertools
import nltk
import numpy as np

In [38]:
import csv
import itertools
import os

import nltk
import numpy as np

# Reproducibility
np.random.seed(42)

vocabulary_size = 8000
unknown_token = "UNKNOWN_TOKEN"
sentence_start_token = "SENTENCE_START"
sentence_end_token = "SENTENCE_END"

# If you are in Google Colab, mount Drive. If not, this block is safely ignored.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

# Adjust this path if your CSV is stored elsewhere in Drive.
drive_file_path = '/content/drive/My Drive/reddit-comments-2015-08.csv'

# Make sure the tokenizer resources exist.
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

if not os.path.exists(drive_file_path):
    raise FileNotFoundError(
        f"Could not find the dataset at: {drive_file_path}\n"
        "Please update 'drive_file_path' to the correct CSV location."
    )

print("Reading CSV file...")
with open(drive_file_path, 'r', encoding='utf-8') as f:
    reader = csv.reader(f, skipinitialspace=True)
    next(reader, None)  # skip header if present

    raw_sentences = []
    for row in reader:
        if not row:
            continue
        text = row[0].strip().lower()
        if not text:
            continue
        raw_sentences.extend(nltk.sent_tokenize(text))

sentences = [
    f"{sentence_start_token} {s} {sentence_end_token}"
    for s in raw_sentences
]

print(f"Parsed {len(sentences)} sentences.")

tokenized_sentences = [nltk.word_tokenize(sent) for sent in sentences]

word_freq = nltk.FreqDist(itertools.chain(*tokenized_sentences))
print(f"Found {len(word_freq)} unique word tokens.")

vocab = word_freq.most_common(vocabulary_size - 1)
index_to_word = [word for word, _ in vocab] + [unknown_token]
word_to_index = {word: i for i, word in enumerate(index_to_word)}

print(f"Using vocabulary size {len(index_to_word)}.")
print(
    f"The least frequent word in the vocabulary is '{vocab[-1][0]}' "
    f"and appeared {vocab[-1][1]} times."
)

for i, sent in enumerate(tokenized_sentences):
    tokenized_sentences[i] = [w if w in word_to_index else unknown_token for w in sent]

print("\nExample sentence:")
print(sentences[0])
print("\nAfter preprocessing:")
print(tokenized_sentences[0])

X_train = [[word_to_index[w] for w in sent[:-1]] for sent in tokenized_sentences]
y_train = [[word_to_index[w] for w in sent[1:]] for sent in tokenized_sentences]

print(f"Built {len(X_train)} training examples.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading CSV file...
Parsed 79184 sentences.
Found 63011 unique word tokens.
Using vocabulary size 8000.
The least frequent word in the vocabulary is 'whitebeard' and appeared 10 times.

Example sentence:
SENTENCE_START i joined a new league this year and they have different scoring rules than i'm used to. SENTENCE_END

After preprocessing:
['SENTENCE_START', 'i', 'joined', 'a', 'new', 'league', 'this', 'year', 'and', 'they', 'have', 'different', 'scoring', 'rules', 'than', 'i', "'m", 'used', 'to', '.', 'SENTENCE_END']
Built 79184 training examples.


### RNN Model Parameters Explained

**Weight Matrices:**

*   `self.Wax`
    *   This is the weight matrix from the input `x` to the hidden state `a`.
    *   Shape: `(hidden_size, vocab_size)`
    *   It tells the RNN how much each input character should affect each hidden neuron.

*   `self.Waa`
    *   This is the weight matrix from the previous hidden state `a(t-1)` to the current hidden state `a(t)`.
    *   Shape: `(hidden_size, hidden_size)`
    *   It is what gives the RNN its memory. It lets the network use what it already knows from earlier time steps.

*   `self.Wya`
    *   This is the weight matrix from the hidden state to the output `y`.
    *   Shape: `(vocab_size, hidden_size)`
    *   It converts the hidden state into prediction scores for each character in the vocabulary.

**Bias Vectors:**

*   `self.ba`
    *   This is the bias for the hidden layer.
    *   Shape: `(hidden_size, 1)`
    *   It shifts the hidden activation values.

*   `self.by`
    *   This is the bias for the output layer.
    *   Shape: `(vocab_size, 1)`
    *   It shifts the output scores before softmax.

**Gradient Storage:**

These are used to store the gradients during backpropagation:

*   `self.dWax`
    *   Gradient of `Wax`

*   `self.dWaa`
    *   Gradient of `Waa`

*   `self.dWya`
    *   Gradient of `Wya`

*   `self.dba`
    *   Gradient of `ba`

*   `self.dby`
    *   Gradient of `by`

They are initialized with zeros so that during the backward pass, updates can be accumulated into them.

**AdamW Optimizer State:**

These variables store the moving averages used by AdamW. Adam keeps two things for each parameter:

*   `m = first moment estimate`
    *   This is the moving average of gradients.
*   `v = second moment estimate`
    *   This is the moving average of squared gradients.

So:

*   `self.mWax, self.vWax`
    *   Adam state for `Wax`

*   `self.mWaa, self.vWaa`
    *   Adam state for `Waa`

*   `self.mWya, self.vWya`
    *   Adam state for `Wya`

*   `self.mba, self.vba`
    *   Adam state for `ba`

*   `self.mby, self.vby`
    *   Adam state for `by`

These are all initialized to zeros because at the start, the model has no past gradient history.

In [39]:
class RNN:
    def __init__(self, hidden_size, vocab_size, learning_rate=0.01, index_to_word=None, word_to_index=None):
        self.hidden_size = hidden_size
        self.vocab_size = vocab_size
        self.learning_rate = learning_rate
        self.index_to_word = index_to_word
        self.word_to_index = word_to_index

        limit_x = np.sqrt(1.0 / vocab_size)
        limit_h = np.sqrt(1.0 / hidden_size)

        self.Wax = np.random.uniform(-limit_x, limit_x, (hidden_size, vocab_size))
        self.Waa = np.random.uniform(-limit_h, limit_h, (hidden_size, hidden_size))
        self.Wya = np.random.uniform(-limit_h, limit_h, (vocab_size, hidden_size))

        self.ba = np.zeros((hidden_size, 1))
        self.by = np.zeros((vocab_size, 1))

        self.reset_optimizer_state()

    def reset_optimizer_state(self):
        self.dWax = np.zeros_like(self.Wax)
        self.dWaa = np.zeros_like(self.Waa)
        self.dWya = np.zeros_like(self.Wya)
        self.dba = np.zeros_like(self.ba)
        self.dby = np.zeros_like(self.by)

        self.mWax = np.zeros_like(self.Wax)
        self.vWax = np.zeros_like(self.Wax)
        self.mWaa = np.zeros_like(self.Waa)
        self.vWaa = np.zeros_like(self.Waa)
        self.mWya = np.zeros_like(self.Wya)
        self.vWya = np.zeros_like(self.Wya)
        self.mba = np.zeros_like(self.ba)
        self.vba = np.zeros_like(self.ba)
        self.mby = np.zeros_like(self.by)
        self.vby = np.zeros_like(self.by)

        self.t = 0

    @staticmethod
    def softmax(x):
        x = x - np.max(x)
        exp_x = np.exp(x)
        return exp_x / np.sum(exp_x)

    def forward(self, X, a_prev):
        x, a, y_pred = {}, {}, {}
        a[-1] = np.copy(a_prev)

        for t in range(len(X)):
            x[t] = np.zeros((self.vocab_size, 1))
            x[t][X[t]] = 1

            a[t] = np.tanh(np.dot(self.Wax, x[t]) + np.dot(self.Waa, a[t - 1]) + self.ba)
            y_pred[t] = self.softmax(np.dot(self.Wya, a[t]) + self.by)

        return x, a, y_pred

    def loss(self, y_pred, targets):
        total_loss = 0.0
        eps = 1e-12
        for t in range(len(targets)):
            total_loss += -np.log(y_pred[t][targets[t], 0] + eps)
        return total_loss

    def backward(self, x, a, y_pred, targets):
        self.reset_gradients()

        da_next = np.zeros((self.hidden_size, 1))

        for t in reversed(range(len(targets))):
            dy = np.copy(y_pred[t])
            dy[targets[t]] -= 1

            self.dWya += np.dot(dy, a[t].T)
            self.dby += dy

            da = np.dot(self.Wya.T, dy) + da_next
            da_raw = (1 - a[t] ** 2) * da

            self.dba += da_raw
            self.dWax += np.dot(da_raw, x[t].T)
            self.dWaa += np.dot(da_raw, a[t - 1].T)

            da_next = np.dot(self.Waa.T, da_raw)

        for grad in [self.dWax, self.dWaa, self.dWya, self.dba, self.dby]:
            np.clip(grad, -5, 5, out=grad)

    def reset_gradients(self):
        self.dWax.fill(0)
        self.dWaa.fill(0)
        self.dWya.fill(0)
        self.dba.fill(0)
        self.dby.fill(0)

    def adamw(self, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=1e-4):
        self.t += 1

        def update(param, grad, m, v):
            m[:] = beta1 * m + (1 - beta1) * grad
            v[:] = beta2 * v + (1 - beta2) * (grad ** 2)
            m_hat = m / (1 - beta1 ** self.t)
            v_hat = v / (1 - beta2 ** self.t)
            param -= self.learning_rate * (m_hat / (np.sqrt(v_hat) + epsilon) + weight_decay * param)

        update(self.Wax, self.dWax, self.mWax, self.vWax)
        update(self.Waa, self.dWaa, self.mWaa, self.vWaa)
        update(self.Wya, self.dWya, self.mWya, self.vWya)
        update(self.ba, self.dba, self.mba, self.vba)
        update(self.by, self.dby, self.mby, self.vby)

    def train(self, X_train, y_train, epochs, print_every, max_examples):
        losses = []
        for epoch in range(epochs):
            epoch_loss = 0.0
            total_examples = len(X_train) if max_examples is None else min(len(X_train), max_examples)
            for i in range(total_examples):
                X = X_train[i]
                y = y_train[i]
                if len(X) == 0 or len(y) == 0:
                    continue
                a_prev = np.zeros((self.hidden_size, 1))
                x, a, y_pred = self.forward(X, a_prev)
                loss = self.loss(y_pred, y)
                self.backward(x, a, y_pred, y)
                self.adamw()
                epoch_loss += loss
                if (i + 1) % print_every == 0:
                    avg_loss = epoch_loss / (i + 1)
                    print(f"Epoch {epoch + 1}, Example {i + 1}, Avg Loss: {avg_loss:.4f}")
            avg_epoch_loss = epoch_loss / max(1, total_examples)
            losses.append(avg_epoch_loss)
            print(f"Finished epoch {epoch + 1}/{epochs} - Avg Loss: {avg_epoch_loss:.4f}")

        return losses

    def predict(self, start_words, max_length=50):
        """
        Generate a sequence of words using the trained RNN, starting from the given start sequence.
        The generation stops when SENTENCE_END token is predicted or max_length is reached.

        Args:
        - start_words: A list of starting words (strings).
        - max_length: Maximum number of words to generate.

        Returns:
        - generated_sentence: A string containing the generated sentence.
        """
        a_prev = np.zeros((self.hidden_size, 1))
        generated_indices = []

        # Process starting words to build initial hidden state
        for word in start_words:
            idx = self.word_to_index.get(word, self.word_to_index[unknown_token])
            generated_indices.append(idx)

            x_one_hot = np.zeros((self.vocab_size, 1))
            x_one_hot[idx] = 1

            a_prev = np.tanh(np.dot(self.Wax, x_one_hot) + np.dot(self.Waa, a_prev) + self.ba)

        # If no start words were provided, start with SENTENCE_START token
        if not generated_indices:
            current_word_idx = self.word_to_index[sentence_start_token]
            generated_indices.append(current_word_idx)
        else:
            current_word_idx = generated_indices[-1]

        # Generate new words
        for _ in range(max_length):
            x_one_hot = np.zeros((self.vocab_size, 1))
            x_one_hot[current_word_idx] = 1

            a = np.tanh(np.dot(self.Wax, x_one_hot) + np.dot(self.Waa, a_prev) + self.ba)
            y_pred = self.softmax(np.dot(self.Wya, a) + self.by)

            predicted_idx = np.random.choice(range(self.vocab_size), p=y_pred.ravel())
            generated_indices.append(predicted_idx)

            if self.index_to_word[predicted_idx] == sentence_end_token:
                break

            current_word_idx = predicted_idx
            a_prev = a

        # Convert indices to words
        generated_words = [self.index_to_word[idx] for idx in generated_indices]

        # Join words to form a sentence, handling special tokens
        sentence = ' '.join(generated_words)
        sentence = sentence.replace(f' {sentence_start_token}', '') # Remove leading start token
        sentence = sentence.replace(f'{sentence_end_token}', '') # Remove end token
        sentence = sentence.replace(' UNKNOWN_TOKEN', '') # Remove UNKNOWN_TOKEN if any
        sentence = sentence.replace(' .', '.') # Clean up punctuation
        sentence = sentence.replace(' ,', ',')
        sentence = sentence.replace(' :', ':')
        sentence = sentence.replace(' ;', ';')
        sentence = sentence.replace(' ?', '?')
        sentence = sentence.replace(' !', '!')

        return sentence.strip()

In [40]:
hidden_size = 160
learning_rate = 0.001

rnn = RNN(
    hidden_size=hidden_size,
    vocab_size=len(index_to_word),
    learning_rate=learning_rate,
    index_to_word=index_to_word,
    word_to_index=word_to_index
)

losses = rnn.train(
    X_train,
    y_train,
    epochs=65,
    print_every=500,
    max_examples=10,   # set to a smaller number first if you want a quick test
)

print("Training complete.")



Finished epoch 1/65 - Avg Loss: 172.3658
Finished epoch 2/65 - Avg Loss: 157.9629
Finished epoch 3/65 - Avg Loss: 115.8120
Finished epoch 4/65 - Avg Loss: 96.1533
Finished epoch 5/65 - Avg Loss: 90.4237
Finished epoch 6/65 - Avg Loss: 88.3076
Finished epoch 7/65 - Avg Loss: 87.2726
Finished epoch 8/65 - Avg Loss: 86.6775
Finished epoch 9/65 - Avg Loss: 86.2411
Finished epoch 10/65 - Avg Loss: 85.8829
Finished epoch 11/65 - Avg Loss: 85.5643
Finished epoch 12/65 - Avg Loss: 85.2845
Finished epoch 13/65 - Avg Loss: 85.0397
Finished epoch 14/65 - Avg Loss: 84.8170
Finished epoch 15/65 - Avg Loss: 84.6015
Finished epoch 16/65 - Avg Loss: 84.3835
Finished epoch 17/65 - Avg Loss: 84.1593
Finished epoch 18/65 - Avg Loss: 83.9216
Finished epoch 19/65 - Avg Loss: 83.6620
Finished epoch 20/65 - Avg Loss: 83.3755
Finished epoch 21/65 - Avg Loss: 83.0550
Finished epoch 22/65 - Avg Loss: 82.6906
Finished epoch 23/65 - Avg Loss: 82.2776
Finished epoch 24/65 - Avg Loss: 81.8110
Finished epoch 25/65 -

### Understanding the RNN Class Implementation

The `RNN` class implements a simple Recurrent Neural Network from scratch, including its forward pass, backward pass (using backpropagation through time), and training with the AdamW optimizer.

#### `__init__(self, hidden_size, vocab_size, learning_rate=0.01, index_to_word=None, word_to_index=None)`

-   **Purpose:** Initializes the RNN model's parameters, optimizer state, and vocabulary mappings.
-   **Key Parameters:**
    -   `hidden_size`: Dimensionality of the hidden state vector.
    -   `vocab_size`: Total number of unique words in the vocabulary.
    -   `learning_rate`: Step size for the optimizer.
    -   `index_to_word`, `word_to_index`: Mappings between word indices and actual words, crucial for text generation.
-   **Weight Initialization:**
    -   `self.Wax` (Input to Hidden): `(hidden_size, vocab_size)`
    -   `self.Waa` (Hidden to Hidden): `(hidden_size, hidden_size)`
    -   `self.Wya` (Hidden to Output): `(vocab_size, hidden_size)`
    -   Weights are initialized with small random values (uniform distribution).
-   **Bias Initialization:**
    -   `self.ba` (Hidden Bias): `(hidden_size, 1)`
    -   `self.by` (Output Bias): `(vocab_size, 1)`
    -   Biases are initialized to zeros.
-   **Optimizer State:** Calls `reset_optimizer_state()` to initialize gradient accumulators (`dWax`, `dWaa`, etc.) and AdamW's moment estimates (`mWax`, `vWax`, etc.) to zeros.

#### `softmax(x)`

-   **Purpose:** Converts a vector of raw scores (logits) into a probability distribution.
-   **Mechanism:** Applies the softmax function: $P(y_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$. It subtracts the maximum value from `x` for numerical stability.

#### `forward(self, X, a_prev)`

-   **Purpose:** Computes the hidden states and output predictions for an input sequence `X`.
-   **Input:**
    -   `X`: A list of word indices for the current sequence.
    -   `a_prev`: The hidden state from the *previous* sequence or an initial zero vector.
-   **Process (for each time step `t` in `X`):**
    1.  **One-Hot Encoding:** Converts the input word index `X[t]` into a one-hot vector `x[t]` of shape `(vocab_size, 1)`.
    2.  **Hidden State Update:** Calculates the current hidden state `a[t]` using the previous hidden state `a[t-1]` and the current input `x[t]`:
        $a^{(t)} = \tanh(W_{ax} x^{(t)} + W_{aa} a^{(t-1)} + b_a)$
    3.  **Output Prediction:** Calculates the probability distribution `y_pred[t]` over the vocabulary for the next word:
        $\hat{y}^{(t)} = \text{softmax}(W_{ya} a^{(t)} + b_y)$
-   **Output:** Returns dictionaries `x`, `a`, `y_pred` containing the one-hot inputs, hidden states, and output probability distributions for all time steps.

#### `loss(self, y_pred, targets)`

-   **Purpose:** Calculates the cross-entropy loss between the predicted probabilities and the true target words.
-   **Mechanism:** For each time step, it computes $-\log(P_{\text{true_word}})$ for the predicted probability of the actual next word and sums them up.

#### `backward(self, x, a, y_pred, targets)`

-   **Purpose:** Computes the gradients of the loss with respect to the model's parameters using Backpropagation Through Time (BPTT).
-   **Process:**
    1.  **`reset_gradients()`:** Zeros out all gradient accumulators (`dWax`, `dWaa`, etc.).
    2.  **Iterates Backward:** Loops through the time steps from last to first (`reversed(range(len(targets))`).
    3.  **Output Layer Gradients:**
        -   Calculates `dy` (gradient of loss w.r.t. `y_pred`): `dy = y_pred[t] - one_hot(targets[t])`.
        -   Accumulates gradients for `self.Wya` and `self.by`.
    4.  **Hidden Layer Gradients:**
        -   Calculates `da` (gradient of loss w.r.t. `a[t]`) by backpropagating `dy` through `Wya` and `da_next` (gradient from future hidden state).
        -   Calculates `da_raw` (gradient before `tanh` activation) using the derivative of `tanh`: `(1 - a[t]**2) * da`.
        -   Accumulates gradients for `self.dba`, `self.dWax`, and `self.dWaa`.
        -   Updates `da_next` for the next backward step.
    5.  **Gradient Clipping:** Clips all accumulated gradients to a range (e.g., -5 to 5) to prevent exploding gradients.

#### `reset_gradients(self)`

-   **Purpose:** Sets all gradient accumulators (`dWax`, `dWaa`, `dWya`, `dba`, `dby`) to zero. This is called at the beginning of each backward pass for a new sequence.

#### `adamw(self, beta1=0.9, beta2=0.999, epsilon=1e-8, weight_decay=1e-4)`

-   **Purpose:** Updates the model's parameters using the AdamW optimization algorithm.
-   **Mechanism:** AdamW combines adaptive learning rates for each parameter with a decoupled weight decay mechanism.
    1.  **Moment Estimates:** Calculates exponentially weighted moving averages of past gradients (`m`, first moment) and past squared gradients (`v`, second moment) for each parameter.
    2.  **Bias Correction:** Applies bias correction to `m` and `v` to account for their initialization at zero.
    3.  **Parameter Update:** Updates each parameter `param` using its corrected moment estimates, the learning rate, and a weight decay term:
        `param -= learning_rate * (m_hat / (sqrt(v_hat) + epsilon) + weight_decay * param)`

#### `train(self, X_train, y_train, epochs, print_every, max_examples)`

-   **Purpose:** Orchestrates the training process for the RNN.
-   **Process:**
    1.  **Epoch Loop:** Iterates for a specified number of `epochs`.
    2.  **Example Loop:** Iterates through `X_train` and `y_train` (input sequences and their targets).
    3.  **Forward Pass:** Calls `self.forward()` to get predictions.
    4.  **Loss Calculation:** Calls `self.loss()` to evaluate the prediction error.
    5.  **Backward Pass:** Calls `self.backward()` to compute gradients.
    6.  **Parameter Update:** Calls `self.adamw()` to update the model parameters.
    7.  **Logging:** Prints average loss periodically and at the end of each epoch.

#### `predict(self, start_words, max_length=50)`

-   **Purpose:** Generates a sequence of words starting from a given seed `start_words`.
-   **Process:**
    1.  **Initialize:** Sets `a_prev` (hidden state) to zeros.
    2.  **Seed Processing:** For each `word` in `start_words`:
        -   Converts the word to its index.
        -   Creates a one-hot input vector.
        -   Performs a forward step to update `a_prev` based on the seed words.
    3.  **Generation Loop:** Iterates up to `max_length` or until `SENTENCE_END` token is predicted:
        -   Creates a one-hot input vector for the `current_word_idx`.
        -   Performs a forward step to get `a` (new hidden state) and `y_pred` (probability distribution for the next word).
        -   Samples the `predicted_idx` from `y_pred` (using `np.random.choice`).
        -   Appends `predicted_idx` to `generated_indices`.
        -   Updates `current_word_idx` and `a_prev`.
    4.  **Post-processing:** Converts `generated_indices` back to words, joins them into a sentence, and cleans up special tokens (`SENTENCE_START`, `SENTENCE_END`, `UNKNOWN_TOKEN`) and punctuation spacing.

In [45]:
generated_sentence = rnn.predict(start_words=['i', 'love'])
print(generated_sentence)

i love no points dumb rules they not clear background highest top the sell per felon check on n't scoring 'm.
